# Sprint 4 — Pipeline RAG e Assistente Conversacional de Troubleshooting

**Challenge FIAP / Forzy — PLN para digital-twin de motores elétricos industriais**

Este notebook cobre:
1. Chunking dos manuais técnicos (preservando cabeçalhos/coerência semântica)
2. Geração de embeddings e indexação vetorial (FAISS)
3. Retriever com re-ranking, filtrado pelo estado operacional do equipamento
4. Assistente conversacional (persona técnica, memória de curto prazo, citação de fonte)
5. Avaliação: faithfulness, answer relevancy, context precision + demonstração em 3 cenários

In [ ]:
# !pip -q install sentence-transformers faiss-cpu pandas rouge-score --upgrade
# Para usar um LLM real (recomendado), configure UMA das opções abaixo antes de rodar a seção 4:
#   - OPENAI_API_KEY (via variável de ambiente) para usar a API da OpenAI/Anthropic-compatible
#   - ou rode um modelo local via Ollama (ex.: `ollama pull llama3`) e defina USE_OLLAMA=True
# Na ausência de qualquer LLM configurado, o notebook cai em um gerador extrativo (fallback),
# que compõe a resposta a partir dos chunks recuperados, mantendo o pipeline 100% executável.

In [ ]:
import json, re, os
from pathlib import Path
import pandas as pd

# Dados embutidos diretamente no notebook para que ele rode de forma autossuficiente
# em qualquer ambiente (ex.: upload avulso no Google Colab, sem a pasta data/ ao lado).
# Se a pasta ../data existir (execução local a partir do repositório), ela tem prioridade.
DATA_DIR = Path('../data')
MANUALS_DIR = DATA_DIR / 'manuals'

MANUAL_MT042 = r"""# Manual Técnico — Motor Elétrico Industrial MT-042 (Linha Trifásica IE3, 75 kW, 380V)

## 1. Especificações Gerais

O motor MT-042 é um motor de indução trifásico de alta eficiência (classe IE3), potência nominal
de 75 kW, tensão nominal 380V, corrente nominal 128A, rotação nominal 1780 RPM, grau de
proteção IP55, classe de isolamento F (temperatura máxima admissível de enrolamento: 155°C,
com elevação limite de 80°C sobre ambiente de 40°C). O baseline operacional de temperatura do
enrolamento em regime nominal é de aproximadamente 60°C.

## 2. Sistema de Refrigeração

O MT-042 utiliza refrigeração forçada (IC411) por meio de ventilador acoplado ao eixo. O sistema
depende de fluxo de ar desobstruído nas grades de admissão e exaustão da carcaça.

### 2.1 Procedimento de Verificação do Sistema de Refrigeração

1. Desligar o motor e aguardar resfriamento (mínimo 15 minutos).
2. Inspecionar visualmente as grades de admissão e exaustão quanto a acúmulo de poeira,
   obstruções ou danos nas palhetas do ventilador.
3. Limpar as aletas da carcaça com ar comprimido de baixa pressão (máx. 2 bar).
4. Verificar folga do rolamento do ventilador; substituir se houver ruído ou folga excessiva.
5. Religar o motor e monitorar a temperatura do enrolamento por 30 minutos, comparando com o
   baseline de 60°C. Um desvio acima de +15°C após a limpeza indica possível problema
   estrutural na refrigeração (motor subdimensionado para a carga, obstrução interna) ou
   sobrecarga elétrica — nesse caso, escalar para inspeção elétrica (seção 4).

## 3. Anomalias Térmicas — Diagnóstico e Ação

| Desvio de Temperatura do Enrolamento | Classificação | Ação Recomendada |
|---|---|---|
| até +10°C acima do baseline | leve | Monitorar; registrar em log |
| +10°C a +20°C acima do baseline | moderado | Verificar sistema de refrigeração (seção 2.1) |
| acima de +20°C acima do baseline | crítico | Parar o motor; inspecionar refrigeração E carga elétrica (sobrecarga, desbalanceamento de fases) |

**Atenção:** temperaturas de enrolamento acima de 140°C (elevação de +80°C sobre o baseline de
60°C) exigem parada IMEDIATA do motor para evitar dano permanente ao isolamento classe F.

## 4. Sistema Elétrico e Isolamento

A resistência de isolamento nominal (medida entre enrolamento e carcaça, com megômetro a 500V)
deve ser igual ou superior a 5 MΩ. Valores abaixo de 1 MΩ indicam risco iminente de curto-circuito
para a carcaça e exigem desligamento imediato do motor até nova medição após secagem/limpeza
do enrolamento.

### 4.1 Procedimento de Inspeção do Isolamento

1. Desenergizar completamente o motor e aplicar bloqueio (LOTO).
2. Desconectar os cabos de alimentação nos terminais do motor.
3. Medir a resistência de isolamento com megômetro (500V DC) entre cada fase e a carcaça
   aterrada.
4. Se resistência < 5 MΩ: verificar umidade/contaminação nos enrolamentos; secar em estufa
   a baixa temperatura (máx 90°C) se aplicável.
5. Se resistência < 1 MΩ: NÃO reenergizar. Encaminhar para rebobinamento ou substituição.
6. Registrar valores medidos no relatório de manutenção corretiva.

### 4.2 Anomalias de Corrente

Corrente de fase acima de 10% do valor nominal de forma sustentada indica sobrecarga mecânica
(carga acoplada excessiva) ou desbalanceamento de tensão de alimentação. Verificar a carga
mecânica acoplada antes de qualquer intervenção elétrica.

## 5. Sistema Mecânico — Vibração e Rolamentos

O limite de vibração aceitável para o MT-042 (conforme ISO 10816-3, classe de máquina rígida)
é de 1.0 mm/s RMS em operação nominal (baseline). Valores entre 1.0 e 2.0 mm/s indicam alerta
moderado; acima de 2.0 mm/s indicam alerta crítico, com risco de falha de rolamento ou
desalinhamento severo.

### 5.1 Procedimento de Inspeção de Vibração

1. Com o motor em operação, medir vibração radial e axial nos mancais dianteiro e traseiro.
2. Se vibração acima do baseline: verificar alinhamento do acoplamento com relógio comparador
   ou laser de alinhamento (tolerância máxima 0.05mm).
3. Verificar fixação da base do motor (parafusos de ancoragem) e desbalanceamento do rotor.
4. Inspecionar rolamentos quanto a ruído, folga radial excessiva e temperatura (limite: 45°C
   baseline, alerta acima de 70°C).
5. Se folga excessiva ou ruído metálico constante: substituir rolamento na próxima janela de
   manutenção programada; se vibração crítica (>2.0mm/s) com ruído: parada imediata.

## 6. Manutenção Preventiva — Cronograma

- **Mensal:** limpeza de grades de ventilação, inspeção visual geral, medição de vibração.
- **Trimestral:** medição de resistência de isolamento, lubrificação de rolamentos (graxa de
  lítio complexo, conforme especificação do fabricante).
- **Anual:** rebalanceamento se necessário, revisão completa de alinhamento, substituição
  preventiva de rolamentos com mais de 20.000 horas de operação.

## 7. Segurança

Todas as intervenções elétricas ou mecânicas exigem bloqueio e etiquetagem (LOTO) e uso de
EPIs adequados (luvas isolantes classe 0, óculos de proteção). Nunca realizar medições elétricas
com o motor energizado sem equipamento de proteção adequado (classe de arco elétrico
apropriada).
"""

MANUAL_FICHA_GERAL = r"""# Ficha de Manutenção — Frota de Motores Elétricos Industriais (MT-005, MT-011, MT-017, MT-029, MT-042)

## 1. Visão Geral da Frota

A frota monitorada é composta por 5 motores de indução trifásicos de potências entre 45kW e
90kW, todos equipados com sensores de temperatura de enrolamento, temperatura de rolamento,
vibração (radial e axial), corrente de fase e resistência de isolamento, com telemetria em
tempo real integrada à plataforma de digital-twin.

| Motor | Potência | Aplicação | Baseline Temp. Enrolamento | Baseline Vibração |
|---|---|---|---|---|
| MT-005 | 55 kW | Bomba centrífuga | 60°C | 1.0 mm/s |
| MT-011 | 45 kW | Compressor | 58°C | 1.1 mm/s |
| MT-017 | 90 kW | Ventilador industrial | 62°C | 1.0 mm/s |
| MT-029 | 55 kW | Esteira transportadora | 59°C | 1.2 mm/s |
| MT-042 | 75 kW | Bomba de processo | 60°C | 1.0 mm/s |

## 2. Classificação de Eventos (Taxonomia para Classificação Textual)

Para fins de categorização automática de eventos registrados no sistema, utilizam-se as
seguintes categorias, mutuamente exclusivas:

1. **manutenção corretiva** — intervenção após falha ou alerta crítico já manifestado
   (ex.: substituição de rolamento danificado, rebobinamento por falha de isolamento).
2. **manutenção preventiva** — intervenção programada ou motivada por desvio leve/moderado
   antes de falha (ex.: lubrificação, limpeza, ajuste de alinhamento).
3. **anomalia elétrica** — desvio em parâmetros elétricos: temperatura de enrolamento,
   corrente de fase, resistência de isolamento.
4. **anomalia mecânica** — desvio em parâmetros mecânicos: vibração (radial/axial),
   temperatura de rolamento.
5. **operação normal** — parâmetros dentro da faixa esperada (desvio leve, sem ação requerida
   além de monitoramento).

## 3. Procedimento Padrão de Resposta a Alertas

### 3.1 Alerta Leve
Registrar em log de monitoramento. Nenhuma ação de campo é necessária, exceto reforço da
frequência de leitura do sensor afetado nas próximas 24h.

### 3.2 Alerta Moderado
Gerar ordem de serviço de verificação (não corretiva) a ser executada em até 48h. Consultar o
manual específico do motor (ex.: motor_mt042_manual.md) para o procedimento de inspeção do
subsistema afetado (refrigeração, isolamento, vibração).

### 3.3 Alerta Crítico
Gerar ordem de serviço de manutenção corretiva com prioridade máxima. Avaliar necessidade de
parada imediata do equipamento conforme os limites de segurança descritos no manual técnico
específico do motor. Notificar o responsável de manutenção e registrar o evento no relatório
semanal de equipamentos em risco.

## 4. Rastreabilidade

Todo relatório de estado operacional gerado automaticamente deve referenciar o `alert_id` e o
`sensor` de origem de cada afirmação, permitindo auditoria e correlação com os dados brutos de
telemetria armazenados no histórico do digital-twin.
"""

MANUAL_DOCS = {
    'motor_mt042_manual.md': MANUAL_MT042,
    'ficha_manutencao_geral.md': MANUAL_FICHA_GERAL,
}

if MANUALS_DIR.exists() and list(MANUALS_DIR.glob('*.md')):
    manual_sources = {f.name: f.read_text(encoding='utf-8') for f in sorted(MANUALS_DIR.glob('*.md'))}
else:
    manual_sources = MANUAL_DOCS

print('Documentos técnicos encontrados:', list(manual_sources.keys()))

## 1. Chunking Inteligente

Estratégia: split por cabeçalhos Markdown (`##`, `###`) para preservar coerência semântica e
contexto de seção; sub-chunks aplicados apenas quando uma seção excede o tamanho-alvo, com
sobreposição para não cortar frases no meio. Cada chunk carrega metadados: documento de origem,
título de seção e motor(es) a que se aplica (extraído do nome do documento / conteúdo).

In [ ]:
def extrair_motor_ids(texto: str, nome_arquivo: str):
    ids = set(re.findall(r'MT-\d{3}', texto))
    ids.update(re.findall(r'MT-\d{3}', nome_arquivo.upper()))
    return sorted(ids) if ids else ['GERAL']

def chunk_por_secao(texto: str, max_chars=900, overlap=150):
    partes = re.split(r'(?m)^(#{1,3} .+)$', texto)
    blocos = []
    header_atual = partes[0].strip()
    i = 1
    while i < len(partes):
        header = partes[i].strip()
        corpo = partes[i+1] if i+1 < len(partes) else ''
        blocos.append((header, corpo.strip()))
        i += 2
    return blocos

def subdividir(texto: str, max_chars=900, overlap=150):
    if len(texto) <= max_chars:
        return [texto]
    partes = []
    start = 0
    while start < len(texto):
        end = min(start + max_chars, len(texto))
        # tenta cortar em fim de frase
        corte = texto.rfind('.', start, end)
        if corte == -1 or corte < start + max_chars * 0.5:
            corte = end
        else:
            corte += 1
        partes.append(texto[start:corte].strip())
        start = max(corte - overlap, corte) if corte == end else corte - overlap
        if corte == end:
            start = end
    return [p for p in partes if p]

chunks = []
for nome_arquivo, texto in manual_sources.items():
    titulo_doc = texto.splitlines()[0].lstrip('# ').strip()
    blocos = chunk_por_secao(texto)
    for header, corpo in blocos:
        if not corpo:
            continue
        subpartes = subdividir(corpo)
        for idx, sp in enumerate(subpartes):
            chunk_texto = f"{header}\n{sp}" if header else sp
            stem = Path(nome_arquivo).stem
            chunks.append({
                'chunk_id': f"{stem}__{header[:30].strip().replace(' ', '_')}__{idx}",
                'documento': nome_arquivo,
                'titulo_documento': titulo_doc,
                'secao': header,
                'texto': chunk_texto,
                'motores': extrair_motor_ids(chunk_texto, nome_arquivo),
            })

chunks_df = pd.DataFrame(chunks)
print(f'Total de chunks gerados: {len(chunks_df)}')
chunks_df[['chunk_id', 'documento', 'secao', 'motores']].head(10)

## 2. Embeddings e Indexação Vetorial (FAISS)

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    import faiss
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers', 'faiss-cpu'])
    from sentence_transformers import SentenceTransformer
    import faiss

import numpy as np

EMBED_MODEL = 'paraphrase-multilingual-MiniLM-L12-v2'  # bom suporte a PT-BR
embedder = SentenceTransformer(EMBED_MODEL)

embeddings = embedder.encode(chunks_df['texto'].tolist(), normalize_embeddings=True, show_progress_bar=True)
embeddings = np.asarray(embeddings, dtype='float32')

index = faiss.IndexFlatIP(embeddings.shape[1])  # produto interno = similaridade de cosseno (vetores normalizados)
index.add(embeddings)
print('Índice FAISS criado com', index.ntotal, 'vetores de dimensão', embeddings.shape[1])

## 3. Retriever com Re-ranking por Relevância Contextual

O retriever busca os top-k chunks por similaridade semântica e aplica um re-ranking que:
- favorece chunks cujo(s) motor(es) coincidem com o `motor_id` do estado operacional atual
  (filtro de contexto), sem excluir chunks gerais (aplicáveis a toda a frota);
- combina o score semântico com um bônus lexical simples (overlap de termos técnicos) como
  segundo sinal de relevância.

In [ ]:
def buscar(query: str, motor_atual: str = None, top_k=5, k_inicial=10):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, idxs = index.search(q_emb, k_inicial)
    candidatos = []
    termos_query = set(re.findall(r'\w+', query.lower()))
    for score, idx in zip(scores[0], idxs[0]):
        if idx == -1:
            continue
        row = chunks_df.iloc[idx]
        termos_chunk = set(re.findall(r'\w+', row['texto'].lower()))
        overlap = len(termos_query & termos_chunk) / max(len(termos_query), 1)
        bonus_motor = 0.15 if motor_atual and motor_atual in row['motores'] else (0.0 if motor_atual and row['motores'] != ['GERAL'] else 0.05)
        score_final = float(score) + 0.1 * overlap + bonus_motor
        candidatos.append({**row.to_dict(), 'score_semantico': float(score), 'score_final': score_final})
    candidatos.sort(key=lambda c: c['score_final'], reverse=True)
    return candidatos[:top_k]

resultado = buscar('temperatura do enrolamento acima do baseline, o que fazer?', motor_atual='MT-042', top_k=3)
for r in resultado:
    print(f"[{r['score_final']:.3f}] {r['documento']} — {r['secao']}\n{r['texto'][:200]}...\n")

### 3.1 Avaliação do Retriever (gabarito manual de perguntas-resposta)

Precisão dos chunks retornados: para cada pergunta do gabarito (`troubleshooting_qa.json`),
verifica-se se algum dos top-k chunks recuperados pertence ao documento/seção esperado
(`fonte` no gabarito).

In [ ]:
QA_JSON_EMBUTIDO = r"""[
  {"id": "Q01", "categoria": "anomalia_eletrica", "pergunta": "A temperatura do enrolamento do MT-042 subiu 18°C acima do baseline. O que devo fazer?", "resposta_referencia": "Desvio de +18°C é classificado como alerta moderado (entre +10°C e +20°C). Deve-se verificar o sistema de refrigeração seguindo o procedimento: desligar o motor, aguardar resfriamento, inspecionar grades de admissão/exaustão, limpar aletas com ar comprimido e verificar folga do rolamento do ventilador.", "fonte": "motor_mt042_manual.md, seção 2.1 e 3"},
  {"id": "Q02", "categoria": "anomalia_eletrica", "pergunta": "Qual o limite de resistência de isolamento antes de exigir desligamento imediato do motor?", "resposta_referencia": "Resistência de isolamento abaixo de 1 MΩ indica risco iminente de curto-circuito e exige desligamento imediato do motor até nova medição após secagem/limpeza do enrolamento.", "fonte": "motor_mt042_manual.md, seção 4"},
  {"id": "Q03", "categoria": "anomalia_eletrica", "pergunta": "O que pode causar corrente de fase acima de 10% do valor nominal de forma sustentada?", "resposta_referencia": "Pode indicar sobrecarga mecânica (carga acoplada excessiva) ou desbalanceamento de tensão de alimentação; deve-se verificar a carga mecânica acoplada antes de qualquer intervenção elétrica.", "fonte": "motor_mt042_manual.md, seção 4.2"},
  {"id": "Q04", "categoria": "anomalia_eletrica", "pergunta": "Quais EPIs são necessários para medir a resistência de isolamento de um motor?", "resposta_referencia": "É necessário bloqueio e etiquetagem (LOTO) e uso de luvas isolantes classe 0 e óculos de proteção; nunca realizar medições com o motor energizado sem proteção adequada.", "fonte": "motor_mt042_manual.md, seção 7"},
  {"id": "Q05", "categoria": "anomalia_eletrica", "pergunta": "Acima de qual temperatura de enrolamento é necessária parada imediata do motor MT-042?", "resposta_referencia": "Temperaturas de enrolamento acima de 140°C (elevação de +80°C sobre o baseline de 60°C) exigem parada imediata para evitar dano permanente ao isolamento classe F.", "fonte": "motor_mt042_manual.md, seção 3"},
  {"id": "Q06", "categoria": "anomalia_mecanica", "pergunta": "A vibração radial do MT-017 está em 2.1 mm/s. Isso é crítico?", "resposta_referencia": "Sim. O limite aceitável é 1.0 mm/s (baseline); valores acima de 2.0 mm/s são classificados como alerta crítico, com risco de falha de rolamento ou desalinhamento severo.", "fonte": "motor_mt042_manual.md, seção 5"},
  {"id": "Q07", "categoria": "anomalia_mecanica", "pergunta": "Qual o procedimento de inspeção de vibração em um motor com alerta de vibração elevada?", "resposta_referencia": "Medir vibração radial e axial nos mancais dianteiro e traseiro, verificar alinhamento do acoplamento (tolerância 0.05mm), verificar fixação da base e desbalanceamento do rotor, e inspecionar rolamentos quanto a ruído e temperatura.", "fonte": "motor_mt042_manual.md, seção 5.1"},
  {"id": "Q08", "categoria": "anomalia_mecanica", "pergunta": "Qual a temperatura limite de alerta para o rolamento do motor?", "resposta_referencia": "O baseline de temperatura do rolamento é 45°C; valores acima de 70°C são considerados alerta.", "fonte": "motor_mt042_manual.md, seção 5.1"},
  {"id": "Q09", "categoria": "anomalia_mecanica", "pergunta": "Quando devo substituir um rolamento imediatamente versus programar a substituição?", "resposta_referencia": "Se houver folga excessiva ou ruído metálico constante, substituir na próxima janela de manutenção programada; se a vibração for crítica (acima de 2.0 mm/s) com ruído, a parada deve ser imediata.", "fonte": "motor_mt042_manual.md, seção 5.1"},
  {"id": "Q10", "categoria": "anomalia_mecanica", "pergunta": "Qual a tolerância máxima de alinhamento do acoplamento do motor?", "resposta_referencia": "A tolerância máxima de alinhamento é 0.05mm, verificada com relógio comparador ou laser de alinhamento.", "fonte": "motor_mt042_manual.md, seção 5.1"},
  {"id": "Q11", "categoria": "manutencao_preventiva", "pergunta": "Com que frequência deve ser feita a medição de resistência de isolamento como manutenção preventiva?", "resposta_referencia": "A medição de resistência de isolamento deve ser feita trimestralmente, junto com a lubrificação de rolamentos.", "fonte": "motor_mt042_manual.md, seção 6"},
  {"id": "Q12", "categoria": "manutencao_preventiva", "pergunta": "Qual o cronograma mensal de manutenção preventiva recomendado?", "resposta_referencia": "Mensalmente deve-se realizar limpeza de grades de ventilação, inspeção visual geral e medição de vibração.", "fonte": "motor_mt042_manual.md, seção 6"},
  {"id": "Q13", "categoria": "manutencao_preventiva", "pergunta": "Quando um rolamento deve ser substituído preventivamente por tempo de uso?", "resposta_referencia": "Rolamentos com mais de 20.000 horas de operação devem ser substituídos preventivamente na revisão anual.", "fonte": "motor_mt042_manual.md, seção 6"},
  {"id": "Q14", "categoria": "manutencao_preventiva", "pergunta": "Qual tipo de graxa deve ser usado na lubrificação dos rolamentos?", "resposta_referencia": "Graxa de lítio complexo, conforme especificação do fabricante, aplicada na lubrificação trimestral dos rolamentos.", "fonte": "motor_mt042_manual.md, seção 6"},
  {"id": "Q15", "categoria": "manutencao_preventiva", "pergunta": "O que fazer diante de um alerta moderado, segundo o procedimento padrão da frota?", "resposta_referencia": "Deve-se gerar uma ordem de serviço de verificação (não corretiva) a ser executada em até 48h, consultando o manual específico do motor para o procedimento de inspeção do subsistema afetado.", "fonte": "ficha_manutencao_geral.md, seção 3.2"},
  {"id": "Q16", "categoria": "geral", "pergunta": "Quais são as cinco categorias usadas para classificar automaticamente os eventos registrados na frota?", "resposta_referencia": "Manutenção corretiva, manutenção preventiva, anomalia elétrica, anomalia mecânica e operação normal.", "fonte": "ficha_manutencao_geral.md, seção 2"},
  {"id": "Q17", "categoria": "geral", "pergunta": "O que deve ser feito diante de um alerta crítico segundo o procedimento padrão?", "resposta_referencia": "Deve-se gerar ordem de serviço de manutenção corretiva com prioridade máxima, avaliar necessidade de parada imediata, notificar o responsável de manutenção e registrar o evento no relatório semanal de equipamentos em risco.", "fonte": "ficha_manutencao_geral.md, seção 3.3"},
  {"id": "Q18", "categoria": "geral", "pergunta": "Qual é o baseline de vibração do motor MT-029?", "resposta_referencia": "O baseline de vibração do MT-029 é 1.2 mm/s.", "fonte": "ficha_manutencao_geral.md, seção 1"},
  {"id": "Q19", "categoria": "geral", "pergunta": "Por que é importante referenciar o alert_id e o sensor de origem nos relatórios automáticos?", "resposta_referencia": "Para garantir rastreabilidade, permitindo auditoria e correlação com os dados brutos de telemetria armazenados no histórico do digital-twin.", "fonte": "ficha_manutencao_geral.md, seção 4"},
  {"id": "Q20", "categoria": "fora_de_escopo", "pergunta": "Qual o preço de mercado atual de um motor MT-042 novo?", "resposta_referencia": "Fora do escopo da documentação técnica disponível; o assistente deve indicar que não possui essa informação nos manuais e sugerir contato com o fabricante ou setor de compras.", "fonte": "não aplicável - fora de escopo"}
]
"""

if (DATA_DIR / 'troubleshooting_qa.json').exists():
    qa = json.loads((DATA_DIR / 'troubleshooting_qa.json').read_text(encoding='utf-8'))
else:
    qa = json.loads(QA_JSON_EMBUTIDO)
qa_df = pd.DataFrame(qa)

def contexto_precision(top_k=3):
    acertos = []
    for item in qa:
        if item['categoria'] == 'fora_de_escopo':
            continue
        fonte_doc = item['fonte'].split(',')[0].strip()
        recuperados = buscar(item['pergunta'], top_k=top_k)
        hit = any(r['documento'] == fonte_doc for r in recuperados)
        acertos.append({'id': item['id'], 'hit': hit, 'documento_esperado': fonte_doc,
                         'documentos_recuperados': [r['documento'] for r in recuperados]})
    return pd.DataFrame(acertos)

precisao_df = contexto_precision(top_k=3)
display(precisao_df)
print(f"Context precision (top-3, nível documento): {precisao_df['hit'].mean()*100:.1f}%")

## 4. Assistente Conversacional de Troubleshooting

Persona: assistente técnico especialista em motores elétricos industriais. Instruído a:
- responder somente com base nos trechos recuperados (RAG);
- citar a fonte (documento + seção) de cada afirmação;
- indicar o nível de confiança (alto/médio/baixo) conforme a cobertura dos documentos;
- considerar o contexto operacional atual (resumo de alerta + estado do equipamento, gerados
  no Sprint 3) injetado no prompt;
- manter memória de curto prazo (histórico da conversa) para diálogos de múltiplos turnos;
- reconhecer quando a pergunta está fora do escopo da documentação disponível.

In [ ]:
SYSTEM_PROMPT = """Você é um assistente técnico especialista em motores elétricos industriais,
atuando dentro de uma plataforma de digital-twin para manutenção preditiva.

Regras obrigatórias:
1. Responda SOMENTE com base nos trechos de documentação técnica fornecidos no contexto.
2. Ao final da resposta, cite a(s) fonte(s) usada(s) no formato [documento — seção].
3. Indique o nível de confiança da resposta (alto, médio ou baixo), com base em quão
   diretamente os trechos recuperados respondem à pergunta.
4. Se a pergunta estiver fora do escopo da documentação disponível, diga isso explicitamente
   e não invente informação (não alucine valores, normas ou procedimentos).
5. Leve em conta o estado operacional atual do equipamento fornecido no contexto, se houver,
   para priorizar recomendações relevantes à situação real.
"""

def montar_prompt(pergunta, chunks_recuperados, contexto_operacional=None, historico=None):
    contexto_docs = '\n\n'.join(
        f"[{c['documento']} — {c['secao']}]\n{c['texto']}" for c in chunks_recuperados
    )
    partes = [SYSTEM_PROMPT]
    if contexto_operacional:
        partes.append(f"### Estado Operacional Atual\n{contexto_operacional}\n")
    if historico:
        hist_txt = '\n'.join(f"{h['papel']}: {h['texto']}" for h in historico[-6:])
        partes.append(f"### Histórico da Conversa\n{hist_txt}\n")
    partes.append(f"### Trechos de Documentação Recuperados\n{contexto_docs}\n")
    partes.append(f"### Pergunta do Operador\n{pergunta}")
    return '\n\n'.join(partes)

In [ ]:
USE_OLLAMA = False  # defina True se tiver Ollama rodando localmente com um modelo baixado

def chamar_llm(prompt: str) -> str:
    if os.environ.get('OPENAI_API_KEY'):
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.2,
        )
        return resp.choices[0].message.content
    if USE_OLLAMA:
        import requests
        r = requests.post('http://localhost:11434/api/generate',
                           json={'model': 'llama3', 'prompt': prompt, 'stream': False})
        return r.json()['response']
    return None  # sinaliza fallback extrativo

def resposta_fallback_extrativo(pergunta, chunks_recuperados):
    """Fallback sem LLM: compõe resposta extrativa a partir do chunk mais relevante,
    citando fonte e atribuindo confiança pelo score semântico do top-1."""
    if not chunks_recuperados:
        return ("Não encontrei informação nos manuais técnicos disponíveis para responder a essa "
                "pergunta. Isso está fora do escopo da documentação atual.\nConfiança: baixa.")
    top = chunks_recuperados[0]
    trecho = top['texto'].split('\n', 1)[-1].strip()
    trecho_curto = trecho[:500] + ('...' if len(trecho) > 500 else '')
    confianca = 'alta' if top['score_semantico'] > 0.55 else ('média' if top['score_semantico'] > 0.35 else 'baixa')
    return (
        f"Com base na documentação técnica disponível: {trecho_curto}\n\n"
        f"Fonte: [{top['documento']} — {top['secao']}]\n"
        f"Confiança: {confianca}."
    )

class AssistenteTroubleshooting:
    def __init__(self):
        self.historico = []  # memória de curto prazo

    def perguntar(self, pergunta: str, motor_atual: str = None, contexto_operacional: str = None, top_k=4):
        recuperados = buscar(pergunta, motor_atual=motor_atual, top_k=top_k)
        prompt = montar_prompt(pergunta, recuperados, contexto_operacional, self.historico)
        resposta = chamar_llm(prompt)
        if resposta is None:
            resposta = resposta_fallback_extrativo(pergunta, recuperados)
        self.historico.append({'papel': 'operador', 'texto': pergunta})
        self.historico.append({'papel': 'assistente', 'texto': resposta})
        return resposta, recuperados

### 4.1 Injeção do Contexto Operacional (saída do Sprint 3)

In [ ]:
from io import StringIO

ALERTS_RAW_CSV_EMBUTIDO = """alert_id,motor_id,timestamp,sensor,parametro,valor_medido,baseline,desvio,unidade,nivel,evento_tipo
A001,MT-042,2026-08-18 08:15:00,temp_enrolamento,temperatura,78,60,18,°C,moderado,anomalia_eletrica
A005,MT-029,2026-08-18 14:00:00,vibracao_axial,vibracao,3.4,1.2,2.2,mm/s,critico,anomalia_mecanica
A011,MT-042,2026-08-19 11:15:00,temp_enrolamento,temperatura,92,60,32,°C,critico,anomalia_eletrica
A013,MT-005,2026-08-19 14:25:00,rolamento_temp,temperatura,70,45,25,°C,critico,anomalia_mecanica
A016,MT-042,2026-08-20 07:30:00,rolamento_temp,temperatura,48,45,3,°C,leve,manutencao_preventiva
A017,MT-017,2026-08-20 08:50:00,temp_enrolamento,temperatura,85,60,25,°C,critico,anomalia_eletrica
A018,MT-005,2026-08-20 10:10:00,vibracao_radial,vibracao,2.4,1.0,1.4,mm/s,critico,anomalia_mecanica
"""

alerts_processed_path = DATA_DIR / 'alerts_processed.csv'
if alerts_processed_path.exists():
    alerts_processed = pd.read_csv(alerts_processed_path)
elif (DATA_DIR / 'alerts_raw.csv').exists():
    alerts_processed = pd.read_csv(DATA_DIR / 'alerts_raw.csv')
    alerts_processed['resumo_gerado'] = 'Resumo não disponível (executar sprint3 primeiro).'
else:
    # Fallback: dados embutidos, usados quando o notebook roda isolado (sem executar o Sprint 3 antes)
    alerts_processed = pd.read_csv(StringIO(ALERTS_RAW_CSV_EMBUTIDO))
    alerts_processed['resumo_gerado'] = (
        'Alerta ' + alerts_processed['nivel'] + ' no Motor ' + alerts_processed['motor_id'] +
        ': ' + alerts_processed['sensor'] + ' com desvio de ' + alerts_processed['desvio'].astype(str) +
        alerts_processed['unidade'] + ' sobre o baseline (executar sprint3_pln_alertas.ipynb para o texto completo).'
    )

def contexto_operacional_motor(motor_id: str) -> str:
    sub = alerts_processed[alerts_processed['motor_id'] == motor_id].tail(3)
    if sub.empty:
        return f'Nenhum alerta recente registrado para {motor_id}.'
    return '\n'.join(f"- {r['resumo_gerado']}" for _, r in sub.iterrows())

print(contexto_operacional_motor('MT-042'))

## 5. Demonstração em Três Cenários de Falha

In [ ]:
assistente = AssistenteTroubleshooting()

cenarios = [
    ('Anomalia elétrica', 'MT-042', 'A temperatura do enrolamento do MT-042 subiu bastante acima do baseline. O que devo fazer?'),
    ('Anomalia mecânica', 'MT-017', 'A vibração radial do MT-017 está crítica. Quais passos de inspeção devo seguir?'),
    ('Manutenção preventiva', 'MT-005', 'Qual o cronograma de manutenção preventiva recomendado e com que frequência medir o isolamento?'),
]

for titulo, motor, pergunta in cenarios:
    print(f'=== Cenário: {titulo} ({motor}) ===')
    print('Pergunta:', pergunta)
    ctx_op = contexto_operacional_motor(motor)
    resposta, recuperados = assistente.perguntar(pergunta, motor_atual=motor, contexto_operacional=ctx_op)
    print('\nContexto operacional injetado:\n', ctx_op)
    print('\nResposta do assistente:\n', resposta)
    print('\n' + '-'*80 + '\n')

## 6. Avaliação do Assistente (20 perguntas de troubleshooting)

Métricas (aproximação sem framework externo tipo RAGAS, para manter execução offline):
- **Faithfulness**: proporção de respostas cujo conteúdo é sustentado pelos chunks recuperados
  (checagem por overlap lexical significativo entre resposta e contexto recuperado).
- **Answer relevancy**: similaridade semântica entre a resposta gerada e a resposta de
  referência (embedding cosine similarity).
- **Context precision**: já calculada na seção 3.1 (nível documento).

> Em ambiente com acesso a LLM-judge (GPT-4/Claude), recomenda-se substituir estas heurísticas
> por avaliação com RAGAS (`faithfulness`, `answer_relevancy`, `context_precision`), citada no
> documento final como próximo passo de robustez.

In [ ]:
from numpy.linalg import norm

def cos_sim(a, b):
    return float(np.dot(a, b) / (norm(a) * norm(b) + 1e-9))

def faithfulness_heuristica(resposta: str, chunks_recuperados) -> float:
    contexto_txt = ' '.join(c['texto'] for c in chunks_recuperados).lower()
    termos_resposta = set(re.findall(r'\w{4,}', resposta.lower()))
    termos_contexto = set(re.findall(r'\w{4,}', contexto_txt))
    if not termos_resposta:
        return 0.0
    return len(termos_resposta & termos_contexto) / len(termos_resposta)

avaliacao = []
assistente_eval = AssistenteTroubleshooting()
for item in qa:
    resposta, recuperados = assistente_eval.perguntar(item['pergunta'], top_k=4)
    emb_resp = embedder.encode([resposta], normalize_embeddings=True)[0]
    emb_ref = embedder.encode([item['resposta_referencia']], normalize_embeddings=True)[0]
    relevancy = cos_sim(emb_resp, emb_ref)
    faith = faithfulness_heuristica(resposta, recuperados)
    avaliacao.append({
        'id': item['id'], 'categoria': item['categoria'],
        'answer_relevancy': relevancy, 'faithfulness': faith,
    })

eval_df = pd.DataFrame(avaliacao)
display(eval_df)
print('\nMédias gerais:')
print(eval_df[['answer_relevancy', 'faithfulness']].mean())
print('\nMédias por categoria:')
print(eval_df.groupby('categoria')[['answer_relevancy', 'faithfulness']].mean())

## 7. Limites do Sistema e Mitigações

- **Perguntas fora do escopo** (ex.: Q20, preço de mercado do motor): o assistente deve
  reconhecer a ausência de suporte documental e não inventar valores. Validado no fallback
  extrativo pelo score semântico baixo (< 0.35) acionando confiança "baixa"; com LLM real,
  reforçado pela regra 4 do `SYSTEM_PROMPT`.
- **Alucinação potencial**: mitigada por (a) instrução explícita de não responder além do
  contexto recuperado, (b) exigência de citação de fonte, (c) heurística de faithfulness
  usada como sinal de auditoria pós-hoc.
- **Cobertura documental limitada**: a base atual cobre apenas MT-042 em detalhe e a frota em
  visão geral; perguntas muito específicas sobre motores sem manual dedicado tendem a cair para
  confiança média/baixa — mitigação: expandir a base de manuais por motor.
- **Reranking simples**: o bônus por motor/lexical é heurístico; em produção recomenda-se um
  cross-encoder (ex.: `ms-marco-MiniLM`) para re-ranking mais robusto.